# IBM Multi-GNN reference run on Kaggle (Phase 4)

Runs IBM's code (github.com/IBM/Multi-GNN, Apache 2.0, pinned at commit `252b025`) **unchanged** on HI-Small with their best GIN setup (`--emlps --reverse_mp --ego --ports`), then writes per-transaction probabilities so `src/` can score them with our metrics.

**Their protocol differs from ours:** their val and test graphs contain every later transaction (neighbour sampling never filters by time), their test runs from Sep 9 to the end (our test **plus** the tail), and they pick the best epoch by val F1 at a 0.5 cutoff.

**Setup:** Add Input → the dataset *IBM Transactions for Anti Money Laundering (AML)* (ealtman2019). Settings: Accelerator **GPU T4 x2**, Internet **on**. Then **Save Version → Save & Run All (Commit)**: it runs in the background with the browser closed. Training stops after `TIME_BUDGET_H` hours so the whole run, including the prediction dump, ends before Kaggle's 12-hour limit.

Outputs, under the version's Output tab: `ibm/preds_val.npz`, `ibm/preds_test.npz`, `ibm/logs/train.log`, `ibm/models/checkpoint_True.tar`.

In [ ]:
import glob, os, time
T0 = time.time()
CSV = glob.glob('/kaggle/input/**/HI-Small_Trans.csv', recursive=True)
assert CSV, 'add the ealtman2019 AML dataset as an input'
CSV = CSV[0]
IBM = '/kaggle/working/ibm'
os.makedirs(f'{IBM}/data/Small_HI', exist_ok=True); os.makedirs(f'{IBM}/models', exist_ok=True); os.makedirs(f'{IBM}/logs', exist_ok=True)
print(CSV, os.path.getsize(CSV) // 2**20, 'MB')

Clone IBM's repo at the pinned commit and install what Kaggle lacks. Their `LinkNeighborLoader` needs `pyg-lib` or `torch-sparse`, which come as wheels matched to the image's torch and CUDA versions. **One compatibility patch** to their code, applied right after the clone: newer PyG makes `BaseTransform.forward` abstract, so their `AddEgoIds` (which only defines `__call__`) can't be instantiated. Renaming that method to `forward` restores their behaviour: PyG's `__call__` now calls `forward` on a shallow copy and returns the result.

In [ ]:
%cd /kaggle/working
!rm -rf Multi-GNN && git clone -q https://github.com/IBM/Multi-GNN.git && cd Multi-GNN && git checkout -q 252b0252afca109d1d216c411c59ff70753b25fc && git log --oneline -1
%cd /kaggle/working/Multi-GNN
!sed -i 's/    def __call__(self, data: Union\[Data, HeteroData\]):/    def forward(self, data: Union[Data, HeteroData]):/' train_util.py && grep -n 'def forward' train_util.py
import torch
v = '.'.join(torch.__version__.split('+')[0].split('.')[:2]) + '.0'
cu = 'cu' + torch.version.cuda.replace('.', '')
print(torch.__version__, cu, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: set the accelerator to GPU T4 x2')
!pip install -q torch_geometric wandb munch
!pip install -q pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{v}+{cu}.html
import torch_geometric, importlib.util
ok = importlib.util.find_spec('pyg_lib') or importlib.util.find_spec('torch_sparse')
print('pyg', torch_geometric.__version__, '| pyg_lib', bool(importlib.util.find_spec('pyg_lib')), '| torch_sparse', bool(importlib.util.find_spec('torch_sparse')))
assert ok, 'neither pyg_lib nor torch_sparse installed: LinkNeighborLoader will fail'

Format the CSV. IBM's `format_kaggle_files.py` needs `datatable`, so this is a line-for-line pandas port (checked against their loop on the first 20,000 rows: all 10 columns identical). It keeps the same first-appearance IDs (sender then receiver; receiving then payment currency), timestamps, columns and stable sort by time. `EdgeID` is the row's position in the raw CSV, equal to our `src_row`.

In [ ]:
import numpy as np, pandas as pd
OUT = f'{IBM}/data/Small_HI/formatted_transactions.csv'
raw = pd.read_csv(CSV, dtype=str)   # duplicate 'Account' header -> 'Account', 'Account.1'
ts = pd.to_datetime(raw['Timestamp'], format='%Y/%m/%d %H:%M')
first = ts.iloc[0].normalize() - pd.Timedelta(seconds=10)
pair_codes = lambda a, b: pd.factorize(np.column_stack([a, b]).ravel())[0].reshape(-1, 2)
cur = pair_codes(raw['Receiving Currency'], raw['Payment Currency'])                        # cur1 = receiving, cur2 = payment
acc = pair_codes(raw['From Bank'] + raw['Account'], raw['To Bank'] + raw['Account.1'])
df = pd.DataFrame({'EdgeID': np.arange(len(raw)), 'from_id': acc[:, 0], 'to_id': acc[:, 1],
                   'Timestamp': (ts - first).dt.total_seconds().astype('int64'),
                   'Amount Sent': raw['Amount Paid'].astype(float).round(6), 'Sent Currency': cur[:, 1],
                   'Amount Received': raw['Amount Received'].astype(float).round(6), 'Received Currency': cur[:, 0],
                   'Payment Format': pd.factorize(raw['Payment Format'])[0], 'Is Laundering': raw['Is Laundering'].astype(int)})
df = df.sort_values('Timestamp', kind='stable')
assert len(df) == 5_078_345 and df['Is Laundering'].sum() == 5177 and df.EdgeID.is_unique
df.to_csv(OUT, index=False)
print(f'{len(df):,} rows, {acc.max() + 1:,} accounts, {cur.max() + 1} currencies -> {OUT}')
del raw, df, ts, cur, acc

import json
json.dump({'paths': {'aml_data': f'{IBM}/data', 'model_to_load': f'{IBM}/models', 'model_to_save': f'{IBM}/models'}},
          open('data_config.json', 'w'), indent=2)

Train with IBM's command for their best Small-HI GIN, plus `--testing` (turns off wandb, which would otherwise ask for a login) and `--save_model --unique_name` (keeps the best-val-F1 checkpoint as `checkpoint_True.tar`; `--unique_name` is a flag in their parser). Their defaults: 100 epochs with no early stop, batch 8,192, 100 + 100 sampled neighbours, settings from `model_settings.json`.

**Time budget:** training is stopped after `TIME_BUDGET_H` hours, measured from the start of the notebook, so the prediction dump below still runs inside Kaggle's 12-hour limit. If it stops early, the log says how many epochs ran, and the write-up states it.

In [ ]:
import subprocess
TIME_BUDGET_H = 10.0
budget = TIME_BUDGET_H * 3600 - (time.time() - T0)
CMD = 'python -u main.py --data Small_HI --model gin --emlps --reverse_mp --ego --ports --testing --save_model --unique_name'
print(time.strftime('%H:%M'), CMD, f'(budget {budget / 3600:.1f} h)', flush=True)
with open(f'{IBM}/logs/train.log', 'w') as log:
    try:
        rc = subprocess.run(CMD, shell=True, stdout=log, stderr=subprocess.STDOUT, timeout=budget).returncode
    except subprocess.TimeoutExpired:
        rc = 'stopped by the time budget'
print(time.strftime('%H:%M'), 'training:', rc)
!echo "epochs done: $(grep -c 'Validation F1' {IBM}/logs/train.log)"; grep -v -i warn {IBM}/logs/train.log | grep -E 'Validation F1|Test F1|split|samples' | tail -6
if not os.path.exists(f'{IBM}/models/checkpoint_True.tar'):
    !echo "---- end of train.log ----"; grep -v -i warn {IBM}/logs/train.log | tail -40
    raise SystemExit('no checkpoint saved: the end of train.log is above')

Dump per-transaction probabilities from the best checkpoint, for val and for their test. Their own `--inference` mode is broken at this commit (it reads `args.avg_tps`, which the parser lacks, and passes `precrec=` to an `evaluate_hetero` that doesn't take it). So this rebuilds the model exactly as their `train_gnn` does and runs their `evaluate_hetero` steps, keeping the class-1 softmax instead of the argmax. Seed edges the sampler drops get no prediction; the coverage is printed.

In [ ]:
import sys, json, numpy as np, pandas as pd, torch, wandb
from torch_geometric.nn import to_hetero
sys.argv = ['main.py', '--data', 'Small_HI', '--model', 'gin', '--emlps', '--reverse_mp', '--ego', '--ports', '--testing', '--unique_name']
from util import create_parser, set_seed
from data_loading import get_data
from train_util import AddEgoIds, extract_param, add_arange_ids, get_loaders
from training import get_model

args = create_parser().parse_args(sys.argv[1:]); set_seed(args.seed)
data_config = json.load(open('data_config.json'))
tr, va, te, tr_i, va_i, te_i = get_data(args, data_config)
wandb.init(mode='disabled', config={k: extract_param(k, args) for k in ['lr', 'n_hidden', 'n_gnn_layers', 'w_ce1', 'w_ce2', 'dropout', 'final_dropout']}
           | {'epochs': args.n_epochs, 'batch_size': args.batch_size, 'model': args.model, 'data': args.data, 'num_neighbors': args.num_neighs, 'loss': 'ce', 'n_heads': None})
config = wandb.config
add_arange_ids([tr, va, te])
tr_l, va_l, te_l = get_loaders(tr, va, te, tr_i, va_i, te_i, AddEgoIds(), args)
device = torch.device('cuda:0')
model = to_hetero(get_model(next(iter(tr_l)), config, args), te.metadata(), aggr='mean')
ck = torch.load(f"{data_config['paths']['model_to_save']}/checkpoint_True.tar", map_location=device, weights_only=False)
model.load_state_dict(ck['model_state_dict']); model.to(device).eval()
print('best checkpoint from epoch', ck['epoch'])
edge_id = pd.read_csv(f'{IBM}/data/Small_HI/formatted_transactions.csv', usecols=['EdgeID']).EdgeID.to_numpy()

# their evaluate_hetero, keeping probabilities and the seed edges' arange ids (= row positions in the formatted file)
def dump(loader, inds, name):
    ids, probs = [], []
    for batch in loader:
        e = batch['node', 'to', 'node']
        seeds = loader.data['node', 'to', 'node'].edge_attr.detach().cpu()[inds.detach().cpu()[e.input_id.detach().cpu()], 0]
        mask = torch.isin(e.edge_attr[:, 0].detach().cpu(), seeds)
        ids.append(e.edge_attr[mask, 0].detach().cpu().long())
        e.edge_attr = e.edge_attr[:, 1:]
        batch['node', 'rev_to', 'node'].edge_attr = batch['node', 'rev_to', 'node'].edge_attr[:, 1:]
        with torch.no_grad():
            batch.to(device)
            out = model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict)[('node', 'to', 'node')]
            probs.append(torch.softmax(out[mask.to(device)], dim=-1)[:, 1].cpu())
    ids, probs = torch.cat(ids).numpy(), torch.cat(probs).numpy()
    keep = np.unique(ids, return_index=True)[1]
    np.savez(f'{IBM}/preds_{name}.npz', src_row=edge_id[ids[keep]], prob=probs[keep])
    print(f'{name}: {len(keep):,} of {len(inds):,} seed edges predicted ({len(keep) / len(inds):.2%}) -> {IBM}/preds_{name}.npz', flush=True)

dump(va_l, va_i, 'val')
dump(te_l, te_i, 'test')

In [ ]:
# keep the saved output small: the predictions, log and checkpoint stay; the 400 MB formatted CSV and the clone go
!rm -f {IBM}/data/Small_HI/formatted_transactions.csv && rm -rf /kaggle/working/Multi-GNN && du -sh {IBM} && ls -la {IBM}
print(f'total {(time.time() - T0) / 3600:.1f} h')